# Customer Churn Forecasting - Logistic Regression
## Module 3: Baseline Linear Classification Model

Logistic Regression serves as the interpretable baseline classifier for churn prediction, trained with balanced class weights to handle the ~26.5% churn imbalance.

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## 3.1 Load and Prepare Data

In [ ]:
# Load and prepare the dataset
if os.path.exists('WA_Fn-UseC_-Telco-Customer-Churn.csv'):
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
elif os.path.exists('telco_customer_churn.csv'):
    df = pd.read_csv('telco_customer_churn.csv')
else:
    raise FileNotFoundError('Dataset not found.')

# Data preprocessing
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'], inplace=True)
df.drop(columns=['customerID'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Feature Engineering
df['AvgMonthlySpend'] = np.where(df['tenure'] > 0, df['TotalCharges'] / df['tenure'], df['MonthlyCharges'])
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = sum((df[col].isin(['Yes', 'Fiber optic', 'DSL'])).astype(int) for col in service_cols if col in df.columns)
df['ChargesPerService'] = np.where(df['TotalServices'] > 0, df['MonthlyCharges'] / df['TotalServices'], df['MonthlyCharges'])
contract_risk = {'Month-to-month': 3, 'One year': 2, 'Two year': 1}
df['ContractRisk'] = df['Contract'].map(contract_risk)

# Encode categorical variables
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
multi_cat_cols = [col for col in cat_cols if df[col].nunique() > 2]
df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True, dtype=int)

print(f'Dataset prepared. Shape: {df.shape}')
print(f'Churn distribution: {df["Churn"].value_counts().to_dict()}')

In [ ]:
# Separate features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train size: {len(X_train):,} | Test size: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean()*100:.2f}% | Test churn rate: {y_test.mean()*100:.2f}%')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3.2 Train Logistic Regression Model

In [ ]:
# Train Logistic Regression with balanced class weights
lr_model = LogisticRegression(
    max_iter=1000, 
    class_weight='balanced', 
    random_state=42,
    solver='lbfgs',
    C=1.0
)
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print('=' * 60)
print('LOGISTIC REGRESSION - RESULTS')
print('=' * 60)
print(f'\nAccuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred)*100:.2f}%')
print(f'Recall:    {recall_score(y_test, y_pred)*100:.2f}%')
print(f'F1-Score:  {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}')

print('\n' + '=' * 60)
print('CLASSIFICATION REPORT')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))

## 3.3 Confusion Matrix

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Retained', 'Churned'],
            yticklabels=['Retained', 'Churned'],
            annot_kws={'size': 16}, linewidths=1, linecolor='white')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Logistic Regression - Confusion Matrix', fontsize=14, fontweight='bold')

# Add percentage annotations
total = cm.sum()
for i in range(2):
    for j in range(2):
        pct = cm[i][j] / total * 100
        ax.text(j + 0.5, i + 0.7, f'({pct:.1f}%)', ha='center', va='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('lr_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Print confusion matrix details
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives (Correct Retained):  {tn:,}')
print(f'False Positives (Wrong Churn Alert): {fp:,}')
print(f'False Negatives (Missed Churners):   {fn:,}')
print(f'True Positives (Correct Churn):      {tp:,}')

## 3.4 ROC Curve

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#e74c3c', lw=2, label=f'Logistic Regression (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.5000)')
plt.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - Logistic Regression', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('lr_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Feature Importance (Coefficients)

In [ ]:
# Feature importance based on coefficients
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0],
    'Abs_Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print('Top 15 Most Important Features (Logistic Regression):')
display(feature_importance.head(15))

# Visualize top 15 features
top_features = feature_importance.head(15)
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in top_features['Coefficient']]

plt.figure(figsize=(10, 7))
plt.barh(range(len(top_features)), top_features['Coefficient'].values, color=colors, edgecolor='black', alpha=0.85)
plt.yticks(range(len(top_features)), top_features['Feature'].values)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Top 15 Feature Coefficients - Logistic Regression', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', lw=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Increases Churn Risk'),
                   Patch(facecolor='#2ecc71', label='Decreases Churn Risk')]
plt.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('lr_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Cross-Validation

In [ ]:
# 5-Fold Stratified Cross-Validation
cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print('5-Fold Cross-Validation (ROC-AUC):')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\n  Mean AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'  This confirms the model generalizes well across different data splits.')